In [ ]:
# [1] Setup & Paths
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATH_TEAM_GAMELOGS = Path("/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00d_featurized/2024-25/teamgamelogs_featurized.parquet")
PATH_BOXSCORES = Path("/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/player_gamelogs.parquet")
PATH_ON = Path("/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_player_on_off__dataset_1.parquet")
PATH_OFF = Path("/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_player_on_off__dataset_2.parquet")
PATH_LINEUPS = Path("/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/dashboards/team_dash_lineups__dataset_1.parquet")


In [ ]:
# [2] Cargar DataFrames

def load_parquet(path: Path):
    if path is None:
        return None
    if not path.exists():
        print(f"⚠️ No se encontró {path}.")
        return None
    return pd.read_parquet(path)


df_team = load_parquet(PATH_TEAM_GAMELOGS)
df_box = load_parquet(PATH_BOXSCORES)
df_on = load_parquet(PATH_ON)
df_off = load_parquet(PATH_OFF)
df_lineups = load_parquet(PATH_LINEUPS)

for name, df in [("team", df_team), ("box", df_box), ("on", df_on), ("off", df_off), ("lineups", df_lineups)]:
    if df is None:
        continue
    print(f"
Dataset {name}: {df.shape}")
    display(df.head())


In [ ]:
# [3] Importar funciones
try:
    from 02_FeatureFunctions import add_lineup_features_in_memory, DEFAULT_LINEUP_CONFIG
except ModuleNotFoundError:
    import importlib.util
    import sys

    module_path = Path.cwd() / "02_processing_data/02a_WL_prediction/02_FeatureFunctions.py"
    spec = importlib.util.spec_from_file_location("feature_functions_02", module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    add_lineup_features_in_memory = module.add_lineup_features_in_memory
    DEFAULT_LINEUP_CONFIG = module.DEFAULT_LINEUP_CONFIG

DEFAULT_LINEUP_CONFIG


In [ ]:
# [4] Construcción in-memory de LINEUP_*
if df_team is None or df_box is None:
    raise RuntimeError("Se requieren teamgamelogs y boxscores para construir las métricas de lineup.")

df_aug = add_lineup_features_in_memory(
    df_teamgames=df_team,
    df_player_box=df_box,
    df_on=df_on,
    df_off=df_off,
    df_lineups=df_lineups,
    config=DEFAULT_LINEUP_CONFIG,
)

lineup_cols = sorted([c for c in df_aug.columns if c.startswith("LINEUP_")])
print(f"Columnas LINEUP detectadas: {len(lineup_cols)}")
display(df_aug[lineup_cols].describe(include='all'))


In [ ]:
# [5] Sanity checks + correlaciones
lineup_na = df_aug.filter(like='LINEUP_').isna().mean().sort_values(ascending=False)
print("Proporción de NaN por columna LINEUP_:")
display(lineup_na)

if 'LINEUP_SCORE' in df_aug.columns:
    ax = df_aug['LINEUP_SCORE'].hist(bins=30, figsize=(6, 4))
    ax.set_title('Distribución de LINEUP_SCORE')
    ax.set_xlabel('LINEUP_SCORE')
    ax.set_ylabel('Frecuencia')
    plt.show()

    if 'WL_NUM' in df_aug.columns:
        corr = df_aug.filter(like='LINEUP_').corrwith(df_aug['WL_NUM']).sort_values(ascending=False)
        print("
Correlación LINEUP_* vs WL_NUM:")
        display(corr)


# [6] Conclusiones
- LINEUP_SCORE queda acotado en [0, 1] gracias a la normalización por percentiles históricos por equipo, lo que facilita la comparación directa entre conjuntos.
- La ponderación por MIN_exp suaviza la varianza entre partidos y refleja la estabilidad del núcleo de rotación antes de cada juego.
- LINEUP_EFF_ADJ solo aparece cuando existen datos on/off válidos; en su ausencia, LINEUP_SCORE sigue disponible sin penalizar la falta de información.
- Las métricas auxiliares (STARTERS_OUT, BENCH_DEPTH, MIN_VAR) permiten diagnosticar disponibilidad y profundidad, dando contexto adicional al puntaje agregado.
